# 01 — Does temporal modeling help?

Runs T0 set pooling, T1 time-aware pooling, T2 ordered GRU, T3 shuffled GRU and T4 gap-aware GRU on the **same frozen monthly vectors**. Every history model predicts a withheld recent month vector using only pre-cutoff unlabeled histories. The LightGBM probe uses the training-period validation month only. No OOT score is inspected here.

T2 versus T3 controls GRU capacity but age tags still reveal chronology; treat it as an order-input test, not proof that timestamps contain no order information. Choose a variant using validation, seed stability and compute, then freeze it before notebook 02.

In [ ]:
from pathlib import Path
import json, sys
import pandas as pd
REPO = Path.cwd()
if not (REPO / 'scr_features').exists(): REPO = REPO.parent
sys.path.insert(0, str(REPO))
WORK = REPO / 'notebook_runs' / 'demo'  # Switch to 'real' after notebook 00 real mode.
stage00 = json.loads((WORK / 'stage00.json').read_text(encoding='utf-8'))
DEMO = stage00['demo']
SEEDS = [17] if DEMO else [17, 29, 43]
EPOCHS = 2 if DEMO else 40
VARIANTS = ['T0', 'T1', 'T2', 'T3', 'T4']
print('Fit cutoff:', json.loads((Path(stage00['record_model'])/'manifest.json').read_text())['fit_max_anchor'])

In [ ]:
from scr_features.hierarchy import fit_history, encode_applications
from scr_features.io import combine_parquet
from scr_features.evaluation import temporal_probe

results = []
runs = {}
for variant in VARIANTS:
    for seed in SEEDS:
        key = f'{variant}_seed{seed}'
        model_dir = WORK / 'models' / 'history' / key
        fit_manifest = fit_history(stage00['fit_months'], stage00['fit_apps'],
            stage00['record_model'], stage00['earliest_application'], str(model_dir),
            variant=variant, epochs=EPOCHS, seed=seed)
        fit_emb = WORK / 'embeddings' / key / 'fit.parquet'
        encode_applications(stage00['fit_months'], stage00['fit_apps'], str(model_dir),
                            str(fit_emb), for_fit=True)
        scored = {}
        encoding_seconds = 0.0
        for group, paths in stage00['groups'].items():
            out = WORK / 'embeddings' / key / f'{group}.parquet'
            score_result = encode_applications(paths['months'], paths['apps'], str(model_dir), str(out))
            encoding_seconds += score_result['encoding_seconds']
            scored[group] = str(out)
        combined = WORK / 'embeddings' / key / 'all_applications.parquet'
        combine_parquet(list(scored.values()), str(combined))
        runs[key] = {'model': str(model_dir), 'fit_applications': str(fit_emb),
                     'scored': scored, 'combined': str(combined),
                     'encoding_seconds': encoding_seconds}
        for product in ('CREDIARIO', 'CONSIG_EP'):
            report = temporal_probe(stage00['incumbent'], {key: str(combined)},
                stage00['baseline_columns'], product=product,
                validation_month=stage00['validation_month'],
                train_provenances=stage00['train_provenances'],
                model_params={'n_estimators': 30, 'min_child_samples': 5} if DEMO else None)
            metrics = report['variants'][key]
            results.append({'variant': variant, 'seed': seed, 'product': product,
                            'validation_gini': metrics['gini'],
                            'missing_rate': metrics['missing_embedding_rate'],
                            'ssl_heldout_loss': fit_manifest['heldout_prediction_loss'],
                            'history_parameters': fit_manifest['parameter_count'],
                            'training_seconds': fit_manifest['training_seconds'],
                            'encoding_seconds': encoding_seconds})
table = pd.DataFrame(results)
display(table.sort_values(['product', 'validation_gini'], ascending=[True, False]))

## Unsupervised seed stability on a pre-cutoff anchor panel

This compares the same historical fit applications across seeds. Pairwise distance ranks and neighbor overlap are unchanged by a pure rotation; Procrustes residual measures departures after the best rigid alignment. These checks do not prove identical downstream behavior, but they flag geometry drift before feature promotion.

In [ ]:
from scr_features.stability import compare_embeddings

stability_results = []
if len(SEEDS) > 1:
    for variant in VARIANTS:
        reference = pd.read_parquet(runs[f'{variant}_seed{SEEDS[0]}']['fit_applications'])
        for seed in SEEDS[1:]:
            candidate = pd.read_parquet(runs[f'{variant}_seed{seed}']['fit_applications'])
            metrics = compare_embeddings(reference, candidate,
                keys=['application_id', 'product'], prefix='h_')
            stability_results.append({'variant': variant, 'seed': seed, **metrics})
    display(pd.DataFrame(stability_results))
else:
    print('One seed in synthetic demo; cross-retrain stability requires multiple seeds.')

## Lock the choice before opening OOT

The code below uses T1 only in synthetic demo mode. For the real run, inspect validation results and cost, set `SELECTED_VARIANT` explicitly, and record the rationale. Seed 17 is the predeclared export seed; other seeds are robustness checks. A model with lower self-supervised loss is not automatically the credit winner.

In [ ]:
SELECTED_VARIANT = 'T1' if DEMO else None
if SELECTED_VARIANT not in VARIANTS:
    raise ValueError('Set SELECTED_VARIANT from train-period validation before proceeding')
chosen = f'{SELECTED_VARIANT}_seed17'
stage01 = {'chosen': chosen, 'runs': runs, 'validation_results': results,
           'stability_results': stability_results,
           'selection_basis': 'synthetic execution check' if DEMO else 'record preregistered validation/cost rationale'}
(WORK / 'stage01.json').write_text(json.dumps(stage01, indent=2), encoding='utf-8')
print('Frozen choice:', chosen)